In [1]:
import os

from dotenv import load_dotenv
from scraper import fetch_website_contents
from IPython.display import Markdown, display
from openai import OpenAI

In [2]:
# Load environment variables in a file called .env

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

# Check the key

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
elif api_key.strip() != api_key:
    print("An API key was found, but it looks like it might have space or tab characters at the start or end - please remove them - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")


API key found and looks good so far!


In [3]:
cricbuzz = fetch_website_contents("https://www.cricbuzz.com/")
# print(ed)
print(cricbuzz)

Cricbuzz | Live Cricket Score, Schedule, News, Stats &amp; Videos  | Cricbuzz.com

Menu
Live Scores
Schedule
Archives
News
Series
Teams
Videos
Rankings
More
MATCHES
ZIM
vs
AUS
-
AUS won
INDW
vs
BANW
-
INDW won
BFW
vs
ECR
-
ECR won
PAKW
vs
SLW
-
SLW won
ABF
vs
JKM
-
Preview
ALL
All
Live Now
Today
INTERNATIONAL
England vs Sri Lanka, 2026
Sri Lanka vs England
3rd T20I
Quadrangular T20I Series in Nigeria, 2026
Nigeria vs Ghana
1st Match
Hong Kong, China tour of Japan 2026
Japan vs Hong Kong, China
2nd T20I
North American Cup, 2026
Cayman Islands vs Bermuda
1st Match
Australia tour of Zimbabwe 2026
Zimbabwe vs Australia
3rd ODI
LEAGUE
CPL 2026
Antigua and Barbuda Falcons vs Jamaica Kingsmen
Final
ETPL 2026
Amsterdam Flames vs Belfast Wolves
Qualifier
Belfast Wolves vs Edinburgh Castle Rockers
Final
DOMESTIC
CSA Provincial One-Day Challenge Division Two 2026-27
South Western Districts vs Mpumalanga Rhinos
4th Match
Northern Cape vs Limpopo
5th Match
Border vs Eastern Storm
6th Match
Australi

In [4]:
system_prompt = """
You are a cricket scorecard summarizer.

When given information about an  cricket match, provide a concise and easy-to-read match summary.

Include:
1. The overall match score, showing both teams' scores, overs, and the match result.
2. The Player of the Match.
3. The main highlights and standout performances, including important batting, bowling, or other key contributions.

Format the response clearly so that the user can quickly understand the final score, who won, who was Player of the Match, and the most important moments of the match.

Do not provide the complete ball-by-ball scorecard. Focus only on the overall score, Player of the Match, and key match highlights.

If the scorecard for the requested match is not yet available because the
match has not started, provide the match's starting details instead,
including the teams, scheduled start time, venue, and competition.
"""

In [5]:
user_prompt_prefix = """
The user wants information about this cricket match:
"""

In [6]:
def messages_for(user_request):
    return [
        {"role": "system", "content": system_prompt},
        {
            "role": "user",
            "content": (
                user_prompt_prefix
                + user_request)
        }
    ]

In [7]:
messages_for(cricbuzz)

[{'role': 'system',
  'content': "\nYou are a cricket scorecard summarizer.\n\nWhen given information about an  cricket match, provide a concise and easy-to-read match summary.\n\nInclude:\n1. The overall match score, showing both teams' scores, overs, and the match result.\n2. The Player of the Match.\n3. The main highlights and standout performances, including important batting, bowling, or other key contributions.\n\nFormat the response clearly so that the user can quickly understand the final score, who won, who was Player of the Match, and the most important moments of the match.\n\nDo not provide the complete ball-by-ball scorecard. Focus only on the overall score, Player of the Match, and key match highlights.\n\nIf the scorecard for the requested match is not yet available because the\nmatch has not started, provide the match's starting details instead,\nincluding the teams, scheduled start time, venue, and competition.\n"},
 {'role': 'user',
  'content': "\nThe user wants info

In [8]:
OLLAMA_BASE_URL = "http://localhost:11434/v1"

ollama = OpenAI(base_url=OLLAMA_BASE_URL, api_key='ollama')

In [10]:
def summarize(url):
    website = fetch_website_contents(url)
    response = ollama.chat.completions.create( 
        model="llama3.2:1b",
        messages = messages_for(website)
    )
    return response.choices[0].message.content

In [11]:
summarize("https://www.cricbuzz.com/")

"Based on the information provided, I can summarize the match details as follows:\n\n\n**Match Summary**\n\nIn the 3rd T20I of the quadrangular T20I Series in Nigeria, 2026, India Women defeated Bangladesh Women by 114 runs.\n\n\n**Match Score**\n\nIndia Women: 195/4 (20 overs)\nBangladesh Women: 81/5 (15.4 overs)\nIndia Women: 1-114, Bangladesh Women: 0-81\n\n\n**Player of the Match**\n\n* India Women: Harmanpreet Kaur\n\n\n**Key Highlights**\n\n* India Women's opening batter Harmanpreet Kaur scored a century, breaking down Bangladesh Women's defensless \n* The match was declared as a high-scoring one in which India Women posted the highest total and Bangladesh Women suffered a disappointing batting display\n* India Women won by 114 runs and moved to the top of the T20I series table."

In [12]:
def display_summary(url):
    summary = summarize(url)
    display(Markdown(summary))

In [13]:
display_summary("https://www.cricbuzz.com/")

I. England vs Sri Lanka, 2026
    * Overall score: 271-8 (50)
    * Player of the Match: *None*
    * Main highlights: 
        * Australia wins by 1 wkt
        * India Women wins their first unofficial ODI against Australia A Women

II. Australia U19 tour of India 2026
    * Overview: 
        * England U19 vs Leicestershire was the first two-official match
        * India U19 vs Australia U19 second unofficial ODI
        * India Women vs Bangladesh Women women's semi-final first unofficial ODI

III. Nigeria vs Ghana 
    * Overview: 
        * Nigeria won their first match in the Quadrangular T20I Series